# LensWord — Text Preprocessing

This notebook takes the cleaned Amazon reviews dataset and prepares it for the
LSTM model. It handles tokenization, vocabulary building, converting words to
numbers, padding sequences to equal length, and splitting the data into
train/validation/test sets. The output is saved as PyTorch tensors ready for
model training.

Before running: make sure `01_EDA_lensword.ipynb` has been run first and that
`amazon_reviews_cleaned.csv` exists inside the `data/` folder.

In [1]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import sys
import os

# Add project root to path so we can import config.py
sys.path.append(os.path.abspath('..'))
from src.config import *

In [2]:
# Load the cleaned dataset from EDA notebook
df = pd.read_csv('../data/amazon_reviews_cleaned.csv')

# Confirm it loaded correctly
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset Shape: (3149, 2)

First 5 rows:


,verified_reviews,sentiment
0,Love my Echo!,Positive
1,Loved it!,Positive
2,"Sometimes while playing a game, you can answer...",Positive
3,I have had a lot of fun with this thing. My 4 ...,Positive
4,Music,Positive


In [3]:
# Convert sentiment labels to numbers
# The model needs numbers not words
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['sentiment'].map(label_map)

# Confirm the mapping
print("Label Mapping:", label_map)
print("\nSample:")
df[['sentiment', 'label']].head(10)

Label Mapping: {'Positive': 2, 'Neutral': 1, 'Negative': 0}

Sample:


,sentiment,label
0,Positive,2
1,Positive,2
2,Positive,2
3,Positive,2
4,Positive,2
5,Positive,2
6,Neutral,1
7,Positive,2
8,Positive,2
9,Positive,2


In [4]:
# Clean the review text
# Remove punctuation, numbers, and extra spaces
# Convert everything to lowercase

def clean_text(text):
    text = str(text).lower()                    # lowercase
    text = re.sub(r'[^a-z\s]', '', text)        # remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip()    # remove extra spaces
    return text

df['cleaned_review'] = df['verified_reviews'].apply(clean_text)

# Show before and after
print("Before:", df['verified_reviews'][2])
print("\nAfter: ", df['cleaned_review'][2])

Before: Sometimes while playing a game, you can answer a question correctly but Alexa says you got it wrong and answers the same as you.  I like being able to turn lights on and off while away from home.

After:  sometimes while playing a game you can answer a question correctly but alexa says you got it wrong and answers the same as you i like being able to turn lights on and off while away from home


In [5]:
# Build a vocabulary from all the words in the dataset
# We assign a unique number to each word

# Tokenize all reviews into words
all_words = []
for review in df['cleaned_review']:
    all_words.extend(review.split())

# Count word frequencies
word_counts = Counter(all_words)
print(f"Total unique words found: {len(word_counts)}")

# Keep only the most common words
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(MAX_VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print(f"\nSample words from vocabulary:")
print(list(word2idx.items())[:10])

Total unique words found: 4338
Vocabulary size: 4340

Sample words from vocabulary:
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('i', 3), ('to', 4), ('it', 5), ('and', 6), ('a', 7), ('my', 8), ('is', 9)]


In [6]:
# Convert each review from words to a list of numbers
# using the vocabulary we just built

def text_to_sequence(text):
    words = text.split()
    sequence = [word2idx.get(word, word2idx['<UNK>']) for word in words]
    return sequence

df['sequence'] = df['cleaned_review'].apply(text_to_sequence)

# Show before and after
print("Review:  ", df['cleaned_review'][0])
print("\nSequence:", df['sequence'][0])

Review:   love my echo

Sequence: [11, 8, 12]


In [7]:
# Pad all sequences to the same length (MAX_SEQ_LENGTH = 50)
# Short reviews get zeros added at the end
# Long reviews get cut off at 50 words

def pad_sequence(sequence, max_length):
    if len(sequence) < max_length:
        # Add zeros at the end to reach max_length
        sequence = sequence + [0] * (max_length - len(sequence))
    else:
        # Cut off at max_length
        sequence = sequence[:max_length]
    return sequence

df['padded_sequence'] = df['sequence'].apply(lambda x: pad_sequence(x, MAX_SEQ_LENGTH))

# Confirm all sequences are the same length
lengths = df['padded_sequence'].apply(len)
print(f"All sequences same length: {lengths.nunique() == 1}")
print(f"Sequence length: {lengths[0]}")
print(f"\nSample padded sequence:")
print(df['padded_sequence'][0])

All sequences same length: True
Sequence length: 50

Sample padded sequence:
[11, 8, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [8]:
# Split data into train, validation and test sets
# 75% training, 15% validation, 10% testing
from sklearn.model_selection import train_test_split

X = np.array(df['padded_sequence'].tolist())
y = np.array(df['label'].tolist())

# First split off 25% for val+test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Split the 25% into 15% val and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

print(f"Training set:   {X_train.shape[0]} reviews")
print(f"Validation set: {X_val.shape[0]} reviews")
print(f"Test set:       {X_test.shape[0]} reviews")


Training set:   2361 reviews
Validation set: 472 reviews
Test set:       316 reviews


In [9]:
# Convert numpy arrays to PyTorch tensors
# This is the format PyTorch needs for training

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("Tensors created successfully!")
print(f"X_train shape: {X_train_tensor.shape}")
print(f"X_val shape:   {X_val_tensor.shape}")
print(f"X_test shape:  {X_test_tensor.shape}")

Tensors created successfully!
X_train shape: torch.Size([2361, 50])
X_val shape:   torch.Size([472, 50])
X_test shape:  torch.Size([316, 50])


In [10]:
# Create PyTorch Datasets and DataLoaders
# DataLoader feeds data to the model in batches during training

from torch.utils.data import TensorDataset, DataLoader

# Wrap tensors into datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created successfully!")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

DataLoaders created successfully!
Training batches:   74
Validation batches: 15
Test batches:       10


In [11]:
# Save all preprocessed data so we can load it in the training notebook
import pickle

# Save tensors
torch.save(X_train_tensor, '../data/X_train.pt')
torch.save(X_val_tensor, '../data/X_val.pt')
torch.save(X_test_tensor, '../data/X_test.pt')
torch.save(y_train_tensor, '../data/y_train.pt')
torch.save(y_val_tensor, '../data/y_val.pt')
torch.save(y_test_tensor, '../data/y_test.pt')

# Save vocabulary
with open('../data/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

print("All preprocessed data saved successfully!")
print("Files saved to data/ folder:")
print("  - X_train.pt, X_val.pt, X_test.pt")
print("  - y_train.pt, y_val.pt, y_test.pt")
print("  - word2idx.pkl")

All preprocessed data saved successfully!
Files saved to data/ folder:
  - X_train.pt, X_val.pt, X_test.pt
  - y_train.pt, y_val.pt, y_test.pt
  - word2idx.pkl


In [12]:
# Final summary of preprocessing
print("=" * 40)
print("PREPROCESSING SUMMARY - LensWord")
print("=" * 40)
print(f"Total Reviews:        {len(df)}")
print(f"Vocabulary Size:      {len(vocab)}")
print(f"Max Sequence Length:  {MAX_SEQ_LENGTH}")
print(f"Training Set:         {X_train_tensor.shape[0]}")
print(f"Validation Set:       {X_val_tensor.shape[0]}")
print(f"Test Set:             {X_test_tensor.shape[0]}")
print(f"Batch Size:           {BATCH_SIZE}")
print(f"Training Batches:     {len(train_loader)}")
print("=" * 40)

PREPROCESSING SUMMARY - LensWord
Total Reviews:        3149
Vocabulary Size:      4340
Max Sequence Length:  50
Training Set:         2361
Validation Set:       472
Test Set:             316
Batch Size:           32
Training Batches:     74


In [13]:
import importlib
import imblearn
print("imblearn version:", imblearn.__version__)

imblearn version: 0.14.2


In [14]:
import sys
print(sys.executable)

C:\Users\Administrator\Desktop\Apeiron_ML_2026\Project 07\lensword\.venv\Scripts\python.exe


In [15]:
# Apply SMOTE to fix class imbalance in training set only
# We never apply SMOTE to validation or test sets
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
labels = ['Negative', 'Neutral', 'Positive']
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

# Apply moderate SMOTE - oversample Negative and Neutral to 800 each
# instead of fully matching Positive's 2055 count
sampling_strategy = {0: 800, 1: 800}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter Moderate SMOTE:")
unique, counts = np.unique(y_train_smote, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

print(f"\nTraining set size before SMOTE: {len(X_train)}")
print(f"Training set size after SMOTE:  {len(X_train_smote)}")


Before SMOTE:
  Negative: 192
  Neutral: 114
  Positive: 2055

After Moderate SMOTE:
  Negative: 800
  Neutral: 800
  Positive: 2055

Training set size before SMOTE: 2361
Training set size after SMOTE:  3655


In [16]:
# Convert SMOTE output to PyTorch tensors and save
X_train_smote_tensor = torch.tensor(X_train_smote, dtype=torch.long)
y_train_smote_tensor = torch.tensor(y_train_smote, dtype=torch.long)

# Save new balanced training tensors
torch.save(X_train_smote_tensor, '../data/X_train_smote.pt')
torch.save(y_train_smote_tensor, '../data/y_train_smote.pt')

print("Balanced training tensors saved!")
print(f"X_train_smote shape: {X_train_smote_tensor.shape}")
print(f"y_train_smote shape: {y_train_smote_tensor.shape}")

Balanced training tensors saved!
X_train_smote shape: torch.Size([3655, 50])
y_train_smote shape: torch.Size([3655])


In [17]:
import torch
X_check = torch.load('../data/X_train_smote.pt')
y_check = torch.load('../data/y_train_smote.pt')

import numpy as np
unique, counts = np.unique(y_check.numpy(), return_counts=True)
labels = ['Negative', 'Neutral', 'Positive']
print("Currently saved on disk:")
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

Currently saved on disk:
  Negative: 800
  Neutral: 800
  Positive: 2055
